# Part 1

### Task 1

In [1]:
import pandas as pd
import numpy as np
import nltk
import re
import spacy as sp

customers = pd.read_csv('customer_messages.csv')

/home/amrmaarouf/.venvs/mlenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
customers.head()

,message_id,message
0,MSG001,"Good morning, My phone is not working :( Than..."
1,MSG002,"Hi, Thanks for your quick response! I need hel..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone..."
3,MSG004,"Hey there, The new update is AMAZING!! Please ..."
4,MSG005,"Dear team, The price is 250 EGP, too expensive..."


In [3]:
customers.duplicated().sum()

np.int64(0)

In [5]:
lengths = customers['message'].str.len()
longest_message = customers.loc[lengths.idxmax(), 'message']
shortest_message = customers.loc[lengths.idxmin(), 'message']

In [ ]:
longest_message, shortest_message

('Hey there, I need help with payment for order 10612. Please check. Email: customer36@example.com Visit https://help.example.com/ticket/10612   Please   help   me.   My backup email is user.name+test@sample.org',
 'Dear team, Thanks! Great service :) Thank you.')

### Task 2

Use Python's re module. Create these columns:
    emails: list of email addresses in each message.
    urls: list of links in each message.
    phones: list of phone numbers in the format 010 + 8 digits.
    numbers: list of digit groups found in each message.
    has_urgent: True if the word urgent appears, ignoring case; otherwise False.


In [10]:
msg = customers['message']
customers['emails'] = msg.str.findall(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b')
customers['urls'] = msg.str.findall(r'https?://\S+')
customers['phones'] = msg.str.findall(r'(?<!\d)01[0125]\d{8}(?!\d)')

cleaned = (msg.str.replace(r'https?://\S+', ' ', regex=True)
                .str.replace(r'\b[\w.%+-]+@[\w.-]+\.\w{2,}\b', ' ', regex=True)
                .str.replace(r'(?<!\d)01[0125]\d{8}(?!\d)', ' ', regex=True))
customers['numbers'] = cleaned.str.findall(r'\d+')

customers['has_urgent'] = msg.str.contains(r'\burgent\b', case=False, na=False)

In [14]:
out = customers.copy()

out = out.drop(columns=['phones', 'numbers'])

for col in ['phones', 'numbers']:
    out[col] = (customers[col]
                .apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
                .astype('string'))

out.to_parquet('customers_modified.parquet', index=False)

### Task 3

Create clean_message using re.sub():
    Replace email addresses with [EMAIL].
    Replace URLs with [URL].
    Replace phone numbers with [PHONE].
    Replace remaining groups of digits with [NUMBER].
    Replace repeated whitespace with one space.
    Remove spaces from the start and end.


In [19]:
#Create clean_message using re.sub():
#Replace email addresses with [EMAIL].
#Replace URLs with [URL].
#Replace phone numbers with [PHONE].
#Replace remaining groups of digits with [NUMBER].
#Replace repeated whitespace with one space.
#Remove spaces from the start and end.

replacements = [
    (r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', '[EMAIL]'),
    (r'https?://\S+', '[URL]'),
    (r'(?<!\d)01[0125]\d{8}(?!\d)', '[PHONE]'),
    (r'\d+', '[NUMBER]'),
    (r'\s+', ' ')
]

for pattern, replacement in replacements:
    customers['message'] = customers['message'].apply(
        lambda x: re.sub(pattern, replacement, str(x))
    )

customers['message'] = customers['message'].str.strip()

customers.to_parquet('customers_modified.parquet', index=False)

### Task 4

Create three short test strings yourself: one with an email, one with a phone number, and one with a URL.

Run your extraction and cleaning functions on each string.

Test a string with no email, URL, or phone.

In 2–3 sentences, explain one limitation of your patterns.


In [20]:
test_strings = [
    "Contact me at test@gmail.com",             
    "Call me at 01012345678",                   
    "Visit https://www.google.com for details", 
    "Hello! Have a nice day."                    
]

for text in test_strings:

    emails = re.findall(replacements[0][0], text)
    urls = re.findall(replacements[1][0], text)
    phones = re.findall(replacements[2][0], text)

    cleaned = text

    for pattern, replacement in replacements:
        cleaned = re.sub(pattern, replacement, cleaned)

    cleaned = cleaned.strip()

    print("Original:", text)
    print("Emails:", emails)
    print("URLs:", urls)
    print("Phones:", phones)
    print("Cleaned:", cleaned)
    print("-" * 50)

Original: Contact me at test@gmail.com
Emails: ['test@gmail.com']
URLs: []
Phones: []
Cleaned: Contact me at [EMAIL]
--------------------------------------------------
Original: Call me at 01012345678
Emails: []
URLs: []
Phones: ['01012345678']
Cleaned: Call me at [PHONE]
--------------------------------------------------
Original: Visit https://www.google.com for details
Emails: []
URLs: ['https://www.google.com']
Phones: []
Cleaned: Visit [URL] for details
--------------------------------------------------
Original: Hello! Have a nice day.
Emails: []
URLs: []
Phones: []
Cleaned: Hello! Have a nice day.
--------------------------------------------------
